# Adding Models And Deploying To Raspberry Pi

This notebook explains how to add more models to the project, what information TVM needs to compile them correctly, and how to run compiled artifacts on a target architecture such as Raspberry Pi.

The examples focus on Raspberry Pi 4 with 64-bit Linux, using the `raspi4_aarch64` target profile. The same ideas apply to other target profiles in `compilation/targets.py`.

## What You Need Before Compiling A Model

Before TVM can compile a model, you need to know these details:

- **Frontend**: PyTorch, TensorFlow/Keras, ONNX, or TFLite.
- **Model object or model file**: a loaded framework model, `.onnx`, or `.tflite` file.
- **Input name**: the graph input name TVM should bind at runtime.
- **Input shape**: for example `1,3,224,224` for NCHW image models or `1,224,224,3` for NHWC models.
- **Input dtype**: usually `float32`.
- **Preprocessing**: resize, crop, layout, normalization, channel order, and dtype conversion.
- **Labels**: optional, but useful for readable classification output.
- **Target profile**: the CPU/GPU/OS ABI you want TVM to compile for.

Most broken deployments come from one of three mismatches: wrong input name, wrong input shape/layout, or compiling for an ABI that does not match the target device.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

COMPILATION = ROOT / 'compilation'
if str(COMPILATION) not in sys.path:
    sys.path.insert(0, str(COMPILATION))

from targets import TARGETS

print('Repository:', ROOT)
print('Available targets:', ', '.join(sorted(TARGETS)))


## Three Ways To Add A Model

### 1. Use An ONNX File

This is often the cleanest path for custom models. Export from your training framework to ONNX, inspect the input name and shape, then compile with `--frontend onnx`.

```bash
python compilation/compile.py \
  --frontend onnx \
  --model-path path/to/model.onnx \
  --input-name input0 \
  --input-shape 1,3,224,224 \
  --target x86_64
```

### 2. Use A TFLite File

TFLite is useful for mobile and embedded TensorFlow models. You still need the input name and shape.

```bash
python compilation/compile.py \
  --frontend tflite \
  --model-path path/to/model.tflite \
  --input-name serving_default_input:0 \
  --input-shape 1,224,224,3 \
  --target x86_64
```

### 3. Add A Named Model To The Compiler

For models you will compile often, add a loader directly to `compilation/compile.py`. Add a branch to the relevant `import_*` function and return `(mod, params, labels)`.


## Adding A PyTorch Model

Add a branch to `import_pytorch` in `compilation/compile.py`:

```python
def import_pytorch(model_name: str, input_name: str, input_shape: tuple):
    ...
    registry = {
        ...
        "my_model": (MyModelClass, MyModelWeights.DEFAULT),
    }
```

Or for a model loaded from a checkpoint, handle it inline before the registry lookup:

```python
    if model_name == "my_model":
        model = MyModelDefinition()
        state = torch.load("path/to/weights.pt", map_location="cpu")
        model.load_state_dict(state)
        model.eval()
        torch.set_grad_enabled(False)
        example = torch.zeros(input_shape, dtype=torch.float32)
        scripted = torch.jit.trace(model, example).eval()
        mod, params = relay.frontend.from_pytorch(scripted, [(input_name, input_shape)])
        return mod, params, ["class_a", "class_b"]
```

Keep the pattern flat. Each frontend function returns `(mod, params, labels)`.


## Inspect An ONNX Model

If you do not know the input name or shape, inspect the ONNX graph before compiling. This cell is a template. Change `ONNX_PATH` to your model path and run it.

In [ ]:
ONNX_PATH = None  # Example: ROOT / 'models' / 'my_model.onnx'

if ONNX_PATH is None:
    print('Set ONNX_PATH to inspect your model inputs.')
else:
    import onnx

    model = onnx.load(str(ONNX_PATH))
    for value in model.graph.input:
        dims = []
        for dim in value.type.tensor_type.shape.dim:
            dims.append(dim.dim_value or dim.dim_param or '?')
        elem_type = value.type.tensor_type.elem_type
        print(f'name={value.name} shape={dims} elem_type={elem_type}')

## Validate Locally Before Cross-Compiling

Always compile and run on `x86_64` first. This separates model/frontend/preprocessing issues from target-device issues.

```bash
python compilation/compile.py \
  --frontend onnx \
  --model-path path/to/model.onnx \
  --input-name input0 \
  --input-shape 1,3,224,224 \
  --target x86_64

python examples/python/run_model.py \
  --artifact-dir examples/artifacts/my_model/x86_64 \
  --image examples/assets/cat.png
```

If this fails, fix the model import, input name, input shape, layout, or preprocessing before trying Raspberry Pi.


## Compile For Raspberry Pi 4 64-bit

After the local x86_64 path works, compile with the Raspberry Pi 4 AArch64 target:

```bash
python compilation/compile.py \
  --frontend onnx \
  --model-path path/to/model.onnx \
  --input-name input0 \
  --input-shape 1,3,224,224 \
  --target raspi4_aarch64
```

This writes artifacts to:

```text
examples/artifacts/<model_name>/raspi4_aarch64/
```

Expected files:

- `model.json`: graph executor graph
- `model.params`: parameter bytes
- `model.so`: compiled AArch64 model library
- `metadata.json`: input name, shape, target, and library filename
- `labels.txt`: optional labels if the model provided them

The compiled `model.so` is for the Pi architecture. Do not try to run it inside the x86_64 devcontainer.

Then cross-compile the TVM runtime for the Pi:

```bash
bash compilation/build_runtime.sh raspi4_aarch64
# Output: examples/artifacts/runtime/raspi4_aarch64/
```


## Copy Files To The Raspberry Pi

Keep the copied files in the same `examples/` layout used in this repository. The Python and C++ examples look for the model artifacts and packaged TVM runtime there, so you do not need target-side `/opt/tvm`, `PYTHONPATH`, or `LD_LIBRARY_PATH`.

Example using `rsync`:

```bash
PI=pi@raspberrypi.local
MODEL_DIR=examples/artifacts/my_model/raspi4_aarch64
RUNTIME_DIR=examples/artifacts/runtime/raspi4_aarch64

rsync -av "$MODEL_DIR"   "$PI:~/research/examples/artifacts/my_model/"
rsync -av "$RUNTIME_DIR" "$PI:~/research/examples/artifacts/runtime/"
rsync -av examples/python "$PI:~/research/examples/"
rsync -av examples/cpp    "$PI:~/research/examples/"
rsync -av examples/assets "$PI:~/research/examples/"
```

Expected target layout:

```text
~/research/examples/
  artifacts/my_model/raspi4_aarch64/model.so
  artifacts/my_model/raspi4_aarch64/model.json
  artifacts/my_model/raspi4_aarch64/model.params
  artifacts/my_model/raspi4_aarch64/metadata.json
  artifacts/runtime/raspi4_aarch64/libtvm_runtime.so
  artifacts/runtime/raspi4_aarch64/include/tvm/runtime/module.h
  artifacts/runtime/raspi4_aarch64/python/tvm/__init__.py
  python/run_model.py
  cpp/CMakeLists.txt
  assets/cat.png
```


## Run The Compiled Model On Raspberry Pi With Python

The Python runner configures TVM from the copied artifact/runtime package. It does not require you to export `PYTHONPATH` or `LD_LIBRARY_PATH`, and it does not require `/opt/tvm` on the Pi.

From the folder that contains `examples/`:

```bash
python3 examples/python/run_model.py \
  --artifact-dir examples/artifacts/my_model/raspi4_aarch64 \
  --image examples/assets/cat.png
```

Use `--layout NHWC` for TensorFlow-style image models if the metadata shape does not make the layout obvious. Use `--no-normalize` only when your model expects raw `0..1` image values instead of ImageNet normalization.


## Run The Compiled Model On Raspberry Pi With C++

The C++ runner compiles against headers from `examples/artifacts/runtime/<target>/include` and links against `examples/artifacts/runtime/<target>/libtvm_runtime.so`. It does not require target-side `/opt/tvm`.

From the folder that contains `examples/`:

```bash
cmake -S examples/cpp -B examples/cpp/build -G Ninja
cmake --build examples/cpp/build

examples/cpp/build/run_tvm_graph \
  --artifact-dir examples/artifacts/my_model/raspi4_aarch64 \
  --image examples/assets/cat.png
```

If you copied the runtime package somewhere outside `examples/artifacts/runtime/<target>/`, pass both locations explicitly:

```bash
cmake -S examples/cpp -B examples/cpp/build -G Ninja \
  -DTVM_RUNTIME_DIR=/path/to/runtime-package \
  -DTVM_INCLUDE_DIR=/path/to/runtime-package/include
cmake --build examples/cpp/build
```


## Optional: Cross-Compile The C++ Runner

If you have a Pi-compatible runtime package and sysroot available in the devcontainer, the runner can be cross-compiled with the AArch64 compiler:

```bash
cmake -S examples/cpp -B examples/cpp/build-aarch64 -G Ninja \
  -DCMAKE_C_COMPILER=aarch64-linux-gnu-gcc \
  -DCMAKE_CXX_COMPILER=aarch64-linux-gnu-g++ \
  -DTVM_RUNTIME_DIR=examples/artifacts/runtime/raspi4_aarch64 \
  -DTVM_INCLUDE_DIR=examples/artifacts/runtime/raspi4_aarch64/include
cmake --build examples/cpp/build-aarch64
```

Copy the resulting runner, the runtime package, and the compiled artifact directory to the Pi before running.


## Troubleshooting Checklist

- **TVM Python import fails**: make sure the full runtime package from `compilation/build_runtime.sh` was copied, including `runtime/<target>/python/tvm`.
- **Compile fails at frontend import**: verify model file, input name, input shape, opset, and unsupported operators.
- **Runtime says input not found**: metadata input name does not match the compiled graph input.
- **Output is nonsense**: check preprocessing, layout (`NCHW` vs `NHWC`), normalization, RGB/BGR order, and label order.
- **Pi cannot load `model.so`**: artifact was compiled for the wrong architecture or the Pi is missing `examples/artifacts/runtime/<target>/libtvm_runtime.so`.
- **C++ runner does not configure**: install `build-essential`, `cmake`, and `ninja-build`, then make sure the full runtime package was copied, including `runtime/<target>/include`.
- **C++ runner does not link**: make sure `libtvm_runtime.so` exists under `examples/artifacts/runtime/<target>/` or pass `-DTVM_RUNTIME_DIR`.
- **x86_64 local run works but Pi run fails**: focus on ABI/runtime package mismatch, target profile, and whether the runtime package is for AArch64.
